# Import Fitbit CSV files

Pick your own Fitbit CSV files, review the mapping, then ingest them into a dataset.

1. `cp .env.example .env` and set `W4H_API_KEY` and `W4H_DATASET_ID`
2. Run the cells in order: **pick files → review mapping → import**

The mapping is built locally from your files' headers and column types (no row values leave this machine, no AI key needed). Nothing is imported until you review and approve it.

In [ ]:
import os
import tempfile
from pathlib import Path

import ipywidgets as widgets
import yaml
from IPython.display import display

from w4h_fitbit_file_csv_importer.client import W4HClient
from w4h_fitbit_file_csv_importer.config import dataset_id
from w4h_fitbit_file_csv_importer.csv_import import import_all
from w4h_fitbit_file_csv_importer.manifest import write_manifest, write_plan
from w4h_fitbit_file_csv_importer.mapping import plan_directory
from w4h_fitbit_file_csv_importer.profile import profile_csv
from w4h_fitbit_file_csv_importer.reconcile import reconcile_manifest

client = W4HClient()
ds_id = dataset_id()
# Uploaded files land here (the same folder the CLI reads via W4H_DATA_DIR).
DATA_DIR = Path(os.environ.get("W4H_DATA_DIR") or tempfile.mkdtemp(prefix="w4h-fitbit-"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("dataset:", ds_id)
print("data dir:", DATA_DIR)

## 1. Pick files

In [ ]:
uploader = widgets.FileUpload(accept=".csv", multiple=True, description="Choose CSVs")
uploader

## 2. Save and profile the selected files

In [ ]:
def uploaded_files(widget):
    """(name, bytes) pairs — ipywidgets 8 gives a tuple of dicts, 7 a dict of dicts."""
    value = widget.value
    items = value.values() if isinstance(value, dict) else value
    return [(f["name"], bytes(f["content"])) for f in items]

files = uploaded_files(uploader)
assert files, "Select at least one .csv file in the cell above first."
for name, content in files:
    (DATA_DIR / Path(name).name).write_bytes(content)  # Path().name drops any folder parts
for path in sorted(DATA_DIR.glob("*.csv")):
    p = profile_csv(path)
    print(f"{path.name}: {dict(zip(p['headers'], p['inferred_types']))}")

## 3. Review the mapping

Corrections the checker already applied are listed with the reason. To change anything, edit `manifest_yaml` in the next cell and run the cell after it.

In [ ]:
manifest = plan_directory(DATA_DIR)
review = reconcile_manifest(manifest)
for c in review["corrections"]:
    print(f"corrected {c['signal_slug']} {c['field']}: {c['reason']}")
print("errors:", review["errors"])
manifest_yaml = yaml.safe_dump(manifest, sort_keys=False)
print(manifest_yaml)

In [ ]:
# Approve: re-check (including any edits to manifest_yaml above) and save the reviewed mapping.
manifest = yaml.safe_load(manifest_yaml)
review = reconcile_manifest(manifest)
assert not review["errors"], review["errors"]
write_manifest(DATA_DIR, manifest)
write_plan(DATA_DIR, manifest, review)
print("mapping approved ->", DATA_DIR / "manifest.yaml")

## 4. Import

`append` adds new rows (safe to re-run; duplicates are skipped). `replace` truncates each signal's table first — do not use it on shared datasets.

In [ ]:
mode = widgets.ToggleButtons(options=["append", "replace"], description="Mode:")
run = widgets.Button(description="Import", button_style="primary")
out = widgets.Output()

def on_click(_):
    with out:
        out.clear_output()
        for r in import_all(client, ds_id, manifest, DATA_DIR, mode=mode.value):
            print(f"{r['signal']}: inserted={r['inserted']} skipped={r['skipped']} -> {r.get('physical_schema')}.{r.get('physical_table')}")

run.on_click(on_click)
display(mode, run, out)